# Spike 0.6d — Tier-1 quantitative-sanity counter-checks on Colab

**Spec reference:** `docs/report-final.md §Phase 0 Spike 0.6d` (2026-05-14 addition; H10 supplement).

**Protocol:** `docs/spike_0_6d_protocol.md`.

**Motivation:** `docs/phase_logs/phase_0_signoff.md` Note 2; `docs/phase_logs/spike_0_6c.md` Note 1.

**What this notebook does:**

1. **Sub-spike 0.6d.1 (gating)** — symmetry + dimensional-force sanity on the existing Spike 0.6c Cell 8 history.csv (already on Drive) PLUS one fresh production-Tier-1 run on the NACA 0012 mesh from Spike 0.6c Cell 6. Pass criterion: cycle-averaged force near zero (symmetry) AND dimensional cycle-peak within ±1 order of magnitude of `m × ω² × r_cm`.
2. **Sub-spike 0.6d.2 (gating)** — render the 2D thin-plate pitching cfg (`configs/su2/thin_plate_2d_pitching.cfg.j2`) at production Tier-1 numerics, run SU2 on a 2D thin-plate mesh, compare inviscid-phase moment against the closed-form Sedov/Newman added-mass moment (±15%).
3. **Sub-spike 0.6d.3 (advisory)** — duplicate the production cfg with `SOLVER = INC_NAVIER_STOKES`, run on the same mesh, compare dimensional cycle-averaged forces (±20%).
4. **Aggregator** — `scripts/run_spike_0_6d.py` reads the three result JSONs and writes `data/spike_0_6d/PASS` iff 0.6d.1 AND 0.6d.2 both passed. 0.6d.3 advisory.
5. **Phase 4 gate check** — `scripts/launch_phase4.py --check` returns 0 once BOTH `data/spike_0_6c/PASS` AND `data/spike_0_6d/PASS` exist.

**Compute budget:** ~5–9 hours total on Colab Pro CPU. Counts under Phase 0; NOT against the 1000-h Phase 4 stop rule.

**Prerequisites:** Spike 0.6c notebook must have run at least through Cell 6 (mesh generation) and Cell 8 (NACA 0012 SU2 run that produced `history.csv` on Drive).

## Cell 1 — Imports + constants

All imports for the notebook live in this single cell, at the top, unconditionally. Re-run this cell whenever the runtime restarts.

In [ ]:
import json
import math
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

from google.colab import drive as colab_drive

REPO_URL = 'https://github.com/gclinger/fan-optimization.git'
REPO_DIR = Path('/content/fan-optimization')
DRIVE_DIR = Path('/content/drive/MyDrive/fan-optimization')

SU2_VERSION = 'v8.0.1'
SU2_BIN = Path('/content/SU2_CFD')

REPO_ROOT = REPO_DIR  # alias used by later cells

print(f'[setup] REPO_DIR  = {REPO_DIR}')
print(f'[setup] DRIVE_DIR = {DRIVE_DIR}')
print(f'[setup] SU2_BIN   = {SU2_BIN}')

## Cell 2 — Clone repo + mount Drive

In [ ]:
colab_drive.mount('/content/drive', force_remount=False)
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'[setup] Drive mounted; cache dir {DRIVE_DIR}')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--quiet'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--quiet'], check=True)
print(f'[setup] repo at {REPO_DIR}')
os.chdir(REPO_DIR)

## Cell 3 — Install Python dependencies

Mirrors the Spike 0.6c install cell; `fanopt` is installed in editable mode so the Spike 0.6d module + scripts are picked up directly from the repo.

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[bo]'],
    cwd=REPO_DIR, check=True,
)
# Make scripts/ importable so the in-notebook calls `import run_spike_0_6d_*` work.
scripts_path = str(REPO_DIR / 'scripts')
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)
src_path = str(REPO_DIR / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print('[setup] fanopt + scripts importable')

## Cell 4 — Imports from fanopt + verify

Confirms the Spike 0.6d module landed and exposes the MACH lock constant.


In [ ]:
import fanopt  # noqa: F401
from fanopt.cfd.spike_0_6d import (
    MACH_UNSTEADY_LOCK,
    compute_added_mass_moment_closed_form_2d_plate,
    compute_dimensional_envelope,
)
from fanopt.cfd.configs import (
    render_unsteady_cfg,
    render_thin_plate_2d_pitching_cfg,
)

print(f'[deps] fanopt OK; MACH_UNSTEADY_LOCK = {MACH_UNSTEADY_LOCK} (Round-9 HIGH-12 / C12)')
print('[deps] Spike 0.6d: 0.6d.1 + 0.6d.2 gating, 0.6d.3 advisory')
print('[deps] Phase 4 launch requires BOTH data/spike_0_6c/PASS AND data/spike_0_6d/PASS')

## Cell 5 — Install SU2 (re-uses 0.6c cache if available)

Same install pattern as the 0.6c notebook — Drive cache → conda-forge fallback.

In [ ]:
SU2_CACHE = DRIVE_DIR / 'su2_cache' / SU2_VERSION

def _su2_works() -> bool:
    if not SU2_BIN.exists():
        return False
    r = subprocess.run([str(SU2_BIN), '--help'], capture_output=True, text=True)
    return r.returncode in (0, 1)

if not _su2_works():
    if (SU2_CACHE / 'SU2_CFD').exists():
        shutil.copy(SU2_CACHE / 'SU2_CFD', SU2_BIN)
        os.chmod(SU2_BIN, 0o755)
        print(f'[su2] restored from Drive cache {SU2_CACHE}')
    else:
        print('[su2] no Drive cache; run the Spike 0.6c notebook Cell 5 first to populate it')

if _su2_works():
    print(f'[su2] {SU2_BIN} is callable')
else:
    print('[su2] WARNING: SU2_CFD not installed; sub_1/sub_2/sub_3 SU2 runs will fail until it is')

## Cell 5b — Recover Spike 0.6c.1 PASS marker from existing Drive history.csv

Post-2026-05-14 the 0.6c.1 runner accepts `--su2-history-csv` as evidence from any prior successful SU2 run that used the same Tier-1 numerics (MACH=1e-9 + REF_DIMENSIONALIZATION). The May 2026 Cell 8 history.csv on Drive qualifies, so we recover the 0.6c.1 PASS marker without re-running SU2.

If the recovery file isn't present, the notebook falls back to generating a probe mesh and running SU2 fresh — but that requires SU2_CFD to be installed (Cell 5).

In [ ]:
# Spike 0.6c.1 gate recovery — write data/spike_0_6c/PASS from prior-run evidence.
prior_history = DRIVE_DIR / 'spike_0_6c' / 'sub_2_run' / 'history.csv'
spike_0_6c_dir = REPO_DIR / 'data' / 'spike_0_6c'
spike_0_6c_dir.mkdir(parents=True, exist_ok=True)

if prior_history.exists():
    print(f'[recovery] using prior-run history.csv as evidence: {prior_history}')
    !python3 scripts/run_spike_0_6c_1.py \
        --su2-history-csv {prior_history} \
        --result-json {spike_0_6c_dir}/sub_1_result.json \
        --marker-dir {spike_0_6c_dir}
else:
    print(f'[recovery] no prior history.csv at {prior_history}; falling back to fresh probe')
    probe_mesh = spike_0_6c_dir / 'meshes' / 'probe.su2'
    if not probe_mesh.exists():
        !python3 scripts/gen_benchmark_meshes.py --kind probe --out {probe_mesh}
    !python3 scripts/run_spike_0_6c_1.py \
        --probe-mesh {probe_mesh} \
        --probe-workdir {spike_0_6c_dir}/sub_1_run \
        --result-json {spike_0_6c_dir}/sub_1_result.json \
        --marker-dir {spike_0_6c_dir}

# Run the 0.6c aggregator -> writes data/spike_0_6c/PASS iff sub_1 passed.
!python3 scripts/run_spike_0_6c.py \
    --sub-1-json {spike_0_6c_dir}/sub_1_result.json \
    --out {spike_0_6c_dir}/results.json \
    --marker-dir {spike_0_6c_dir}


## Cell 6 — Sub-spike 0.6d.1 (symmetry + dimensional sanity)

Re-uses the existing Spike 0.6c Cell 8 history.csv on Drive (already produced by the 2026-05-14 0.6c.2 run that triggered the deferral). The envelope is computed for the NACA 0012 geometry that history.csv was generated on.

**F4 default (2026-05-14):** NACA 0012 chord = 1.0 m, density × volume gives effective mass ~0.05 kg, r_cm ~ 0.25 m. The exact NACA 0012 added-mass envelope is dominated by the displaced-air added mass `m_a = π ρ_air b²` per unit span — for c=1 m, b=0.5 m, ρ=1.225 kg/m³, that's 0.96 kg/m. Multiplied by ω² r_cm with ω=12.5664, r=0.25: F_envelope ≈ 0.96 × 158 × 0.25 ≈ 38 N/m. **Adjust these values per the actual mesh dimensions used.**

In [ ]:
# Re-use the existing 0.6c Cell 8 history.csv (already on Drive from the 2026-05-14 run).
existing_history = DRIVE_DIR / 'spike_0_6c' / 'sub_2_run' / 'history.csv'
if not existing_history.exists():
    raise FileNotFoundError(
        f'expected Spike 0.6c Cell 8 history.csv at {existing_history} — '
        'run the Spike 0.6c notebook Cell 8 first.'
    )

result_dir = REPO_DIR / 'data' / 'spike_0_6d'
result_dir.mkdir(parents=True, exist_ok=True)

# NACA 0012 effective mass + r_cm — replace with values measured from the actual
# mesh if these defaults don't match Cell 6's mesh.
naca0012_mass_kg = 0.05
naca0012_r_cm_m = 0.25
omega_rad_per_s = 12.5664  # Tier-1 lock

!python3 scripts/run_spike_0_6d_1.py \
    --history-csv {existing_history} \
    --n-cycles 5 \
    --mass-kg {naca0012_mass_kg} \
    --omega-rad-per-s {omega_rad_per_s} \
    --r-cm-m {naca0012_r_cm_m} \
    --envelope-geometry 'NACA 0012 from Spike 0.6c Cell 6 (chord=1m, m=0.05kg, r_cm=0.25m)' \
    --result-json {result_dir}/sub_1_result.json \
    --marker-dir {result_dir}

## Cell 7 — Sub-spike 0.6d.2 (2D thin-plate added-mass)

Generates a 2D thin-plate mesh, renders the cfg at production Tier-1 numerics (MACH=1e-9 + REF_DIMENSIONALIZATION + LOW_MACH_PREC + DUAL_TIME_STEPPING), runs SU2 for 5 cycles, compares inviscid-phase moment with the closed-form Sedov/Newman value within ±15%.

**Note:** This cell skips the mesh-generation step if you supply a pre-built 2D thin-plate mesh on Drive. Mesh generation logic is operator-supplied (gmsh recipe documented in `docs/spike_0_6d_protocol.md`).

In [ ]:
thin_plate_dir = DRIVE_DIR / 'spike_0_6d' / 'thin_plate_2d'
thin_plate_dir.mkdir(parents=True, exist_ok=True)
thin_plate_mesh = thin_plate_dir / 'thin_plate_2d.su2'

if not thin_plate_mesh.exists():
    print(f'[0.6d.2] thin-plate mesh missing at {thin_plate_mesh}; generating via gmsh')
    !python3 scripts/gen_benchmark_meshes.py \
        --kind thin_plate_2d \
        --out {thin_plate_mesh} \
        --farfield-radius-chord 50.0 \
        --verbose
    if not thin_plate_mesh.exists():
        raise FileNotFoundError(f'mesh generation failed; expected {thin_plate_mesh}')

CHORD_M = 1.0
PIVOT_OFFSET_NORMALIZED = -0.5  # quarter-chord
OMEGA = 12.5664  # Tier-1 lock
THETA_MAX = 0.6981  # Tier-1 lock
T_PERIOD = 2.0 * math.pi / OMEGA
DT = T_PERIOD / 200.0
N_CYCLES = 5
TIME_ITER = N_CYCLES * 200

if thin_plate_mesh.exists():
    cfg_text = render_thin_plate_2d_pitching_cfg(
        mesh_filename=str(thin_plate_mesh),
        marker_plate='PLATE',
        marker_farfield='FARFIELD',
        pitching_omega_y=-abs(OMEGA),  # C11 sign
        pitching_ampl_y=THETA_MAX,
        motion_origin_x=0.25 * CHORD_M,
        time_step=DT,
        max_time=N_CYCLES * T_PERIOD,
        time_iter=TIME_ITER,
    )
    cfg_path = thin_plate_dir / 'thin_plate_2d_pitching.cfg'
    cfg_path.write_text(cfg_text)
    print(f'[0.6d.2] cfg -> {cfg_path}')

    log_path = thin_plate_dir / 'su2.log'
    with log_path.open('w') as logf:
        proc = subprocess.Popen(
            [str(SU2_BIN), str(cfg_path)],
            cwd=thin_plate_dir,
            stdout=logf, stderr=subprocess.STDOUT,
        )
        while proc.poll() is None:
            time.sleep(60)
            tail = subprocess.run(
                ['tail', '-1', str(log_path)], capture_output=True, text=True
            ).stdout.strip()
            print(f'[0.6d.2 heartbeat] {tail[:160]}')
    print(f'[0.6d.2] SU2 exit={proc.returncode}')

history_csv = thin_plate_dir / 'history.csv'
result_dir = REPO_DIR / 'data' / 'spike_0_6d'
result_dir.mkdir(parents=True, exist_ok=True)

if history_csv.exists():
    !python3 scripts/run_spike_0_6d_2.py \
        --history-csv {history_csv} \
        --chord-m {CHORD_M} \
        --pivot-offset-normalized {PIVOT_OFFSET_NORMALIZED} \
        --pitching-omega-rad-per-s {OMEGA} \
        --pitching-amplitude-rad {THETA_MAX} \
        --n-cycles {N_CYCLES} \
        --result-json {result_dir}/sub_2_result.json \
        --marker-dir {result_dir}
else:
    print(f'[0.6d.2] no history.csv at {history_csv}; SU2 likely did not run')

## Cell 8 — Sub-spike 0.6d.3 (incompressible cross-check, advisory)

Optional. Run only if SU2 is built with the incompressible solver enabled. Duplicates the production Tier-1 cfg with `SOLVER = INC_NAVIER_STOKES`, same mesh, motion. Result is advisory; failure does not block Phase 4.

In [ ]:
# This cell is a SKETCH for the operator — actual incompressible-mode cfg generation
# is left as a deferred sub-task (the renderer can produce one but you may want to
# tune INC_NONDIM and reference values manually). Run only if you have a working
# incompressible-mode cfg AND its history.csv. Skip in V1 to keep the gate moving;
# Phase 5 step 62.5 (3-solver cross-check including OpenFOAM) absorbs this work.

comp_history = DRIVE_DIR / 'spike_0_6c' / 'sub_2_run' / 'history.csv'
incomp_history = DRIVE_DIR / 'spike_0_6d' / 'incompressible' / 'history.csv'

result_dir = REPO_DIR / 'data' / 'spike_0_6d'
if comp_history.exists() and incomp_history.exists():
    !python3 scripts/run_spike_0_6d_3.py \
        --comp-history-csv {comp_history} \
        --incomp-history-csv {incomp_history} \
        --result-json {result_dir}/sub_3_result.json
else:
    print('[0.6d.3] SKIPPED — advisory; sub_3 will be marked "not run" by aggregator')
    print(f'  comp history present:   {comp_history.exists()}')
    print(f'  incomp history present: {incomp_history.exists()}')

## Cell 9 — Aggregator + Phase 4 launch gate check

Runs the 0.6d aggregator; writes `data/spike_0_6d/PASS` iff 0.6d.1 AND 0.6d.2 both passed. Then checks the dual-gate (`launch_phase4.py --check`) for end-to-end readiness.

In [ ]:
result_dir = REPO_DIR / 'data' / 'spike_0_6d'
sub_3_arg = []
if (result_dir / 'sub_3_result.json').exists():
    sub_3_arg = ['--sub-3-json', str(result_dir / 'sub_3_result.json')]

!python3 scripts/run_spike_0_6d.py \
    --sub-1-json {result_dir}/sub_1_result.json \
    --sub-2-json {result_dir}/sub_2_result.json \
    {' '.join(sub_3_arg)} \
    --out {result_dir}/results.json \
    --marker-dir {result_dir}

print()
print('=== DUAL-GATE READINESS CHECK ===')
!python3 scripts/launch_phase4.py --check

result = json.loads((result_dir / 'results.json').read_text())
print('\n=== FINAL ===')
print(f"overall_passed: {result['result']['overall_passed']}")
print(f"sub_06d_1.passed: {result['result']['sub_06d_1']['passed']}")
print(f"sub_06d_2.passed: {result['result']['sub_06d_2']['passed']}")
if result['result']['sub_06d_3'] is not None:
    print(f"sub_06d_3.passed (advisory): {result['result']['sub_06d_3']['passed']}")
else:
    print('sub_06d_3: not run (advisory)')

## Cell 10 — (Optional) commit + push the PASS marker

Only run this if you want the `data/spike_0_6d/PASS` marker committed back to the repo. Requires a GitHub PAT with `repo` scope stored in Colab Secrets as `GITHUB_PAT`.

In [ ]:
from google.colab import userdata as colab_userdata

PAT = None
try:
    PAT = colab_userdata.get('GITHUB_PAT')
except Exception as e:
    print(f'[commit] GITHUB_PAT secret not configured ({e}); skipping push')

if PAT:
    subprocess.run(['git', '-C', str(REPO_DIR), 'add', 'data/spike_0_6d/'], check=True)
    subprocess.run(
        ['git', '-C', str(REPO_DIR), 'commit', '-m', 'Spike 0.6d: PASS markers from Colab run'],
        check=False,  # might be no-op if nothing changed
    )
    subprocess.run(
        ['git', '-C', str(REPO_DIR), 'push',
         f'https://{PAT}@github.com/gclinger/fan-optimization.git'],
        check=True,
    )
    print('[commit] pushed')
else:
    print('[commit] skipped (no PAT)')

## Done

If 0.6d.1 + 0.6d.2 both passed AND 0.6c.1 passed previously, `scripts/launch_phase4.py --check` returns 0 — Phase 4 is unblocked. Move on to Phase 1 / Phase 2 / Phase 3 / Phase 4 as the plan dictates.